# doodleai: train the drawing classifier

Runs on **Kaggle** (recommended, free GPU) and on **Google Colab**. All logic lives in
`model/training/*.py`; this notebook only sets paths and calls those scripts.

1. Turn on a GPU (Kaggle: *Settings → Accelerator → GPU*; Colab: *Runtime → Change runtime type → T4 GPU*).
2. Run all cells. Preparing the data takes a while; training resumes automatically if the session is interrupted (run the notebook again).
3. Take `best.keras` and `metrics.json` from the last cell.

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/alanszuster/doodleai"
REF = "main"  # branch, tag or commit to train from
QUICK = True  # True: 5 classes, 2 epochs, a few minutes (checks that everything works)
#              False: the real run (all 120 classes, a few hours)

ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK = "/content/work"  # fast local disk for the big arrays
    RUNS = "/content/drive/MyDrive/doodleai/runs"  # checkpoints survive the session
elif ON_KAGGLE:
    WORK = "/kaggle/temp/doodleai"  # temp disk, not part of the saved output
    RUNS = "/kaggle/working/runs"  # saved when the session is committed
else:
    WORK, RUNS = os.path.abspath("work"), os.path.abspath("work/runs")
if QUICK:  # keep the trial files apart from the real run
    WORK, RUNS = WORK + "-quick", RUNS + "-quick"
PREPARE_ARGS = "--classes 5 --samples 2000" if QUICK else ""
TRAIN_ARGS = "--epochs 2" if QUICK else ""
os.makedirs(WORK, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)

import tensorflow as tf

print("GPUs:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "No GPU: turn one on before training."

In [ ]:
if not os.path.exists("doodleai"):
    subprocess.run(["git", "clone", REPO, "doodleai"], check=True)
subprocess.run(["git", "-C", "doodleai", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", "doodleai", "checkout", "-q", REF], check=True)
subprocess.run(["git", "-C", "doodleai", "pull", "-q", "--ff-only"], check=False)
os.chdir("doodleai")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 1. Prepare the data
Downloads Quick Draw strokes and renders them like the website does. Skipped when already done.

In [ ]:
if not os.path.exists(f"{WORK}/data/meta.json"):
    !python model/training/prepare.py --work-dir {WORK} {PREPARE_ARGS}
else:
    print("Data already prepared.")

In [ ]:
from IPython.display import Image, display

display(Image(f"{WORK}/data/samples.png"))  # what the model will see

## 2. Train
Resumes from the last epoch if this run was interrupted.

In [ ]:
!python model/training/train.py --work-dir {WORK} --runs-dir {RUNS} --name main --resume {TRAIN_ARGS}

## 3. Evaluate on the held-out test split

In [ ]:
!python model/training/evaluate.py --model {RUNS}/main/best.keras --data-dir {WORK}/data --out {RUNS}/main/report

In [ ]:
import json

metrics = json.load(open(f"{RUNS}/main/report/metrics.json"))
print({k: round(metrics[k], 4) for k in ("top1", "top3", "macro_f1", "ece")})
worst = sorted(metrics["per_class_accuracy"].items(), key=lambda kv: kv[1])[:10]
print("Hardest classes:", [(c, round(a, 2)) for c, a in worst])
print("Most confused:", metrics["most_confused"][:5])
display(Image(f"{RUNS}/main/report/confusion_matrix.png"))

## 4. Take the model with you

In [ ]:
import shutil

bundle = shutil.make_archive("/tmp/doodleai-model", "zip", f"{RUNS}/main", ".")
shutil.copy(bundle, f"{RUNS}/doodleai-model.zip")
print(f"{RUNS}/doodleai-model.zip  (best.keras, history.csv, config.json, report/)")
if ON_COLAB:
    from google.colab import files

    files.download(bundle)